# Side Effect Hybrid Seed44

Historical final seed-44 hybrid parameter workflow. Requires the saved calibration-selected parameter files and exact feature matrices referenced by the code. It is not interchangeable with baseline training.


## Stage 1

Source: `side_effect_and_baseline_history`, cell 20 (zero-based).


In [ ]:
# ============================================================
# STAGE 2 — FINAL HYBRID LIGHTGBM
# SEED 44 — GOOGLE COLAB
# FINAL UNIQUE-263 SIDE-EFFECT MODEL
#
# 140 labels -> BASELINE parameters
# 123 labels -> TUNED parameters
#
# Uses:
#   Full original Train
#   Original Calibration for early stopping
#   Original Test probabilities saved only
#
# IMPORTANT:
#   NO TEST METRICS are calculated here.
#
# Resume-safe:
#   If model + metadata already exist,
#   that label is loaded instead of retrained.
# ============================================================


# ============================================================
# 0) INSTALL LIGHTGBM 4.6.0
# ============================================================

import sys
import subprocess
import importlib.metadata

TARGET_LGB_VERSION = "4.6.0"

try:
    installed_version = importlib.metadata.version("lightgbm")
except:
    installed_version = None

if installed_version != TARGET_LGB_VERSION:

    print(
        f"Installing LightGBM {TARGET_LGB_VERSION} "
        f"(current={installed_version})..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"lightgbm=={TARGET_LGB_VERSION}"
    ])


# ============================================================
# 1) IMPORTS
# ============================================================

from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import lightgbm as lgb

import json
import time
import gc
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# 2) MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 3) CONFIG
# ============================================================

SEED = 44

N_LABELS = 263

EARLY_STOPPING_ROUNDS = 50

N_JOBS = 2


# ============================================================
# 4) PATHS
# ============================================================

MYDRIVE = Path(
    "/content/drive/MyDrive"
)


PROJECT_ROOT = (
    MYDRIVE
    / "DDI_SIDE_EFFECT_84_CORRECTED"
)


# IMPORTANT:
# FINAL 263-UNIQUE FEATURE DIRECTORY
FEATURE_DIR = (
    PROJECT_ROOT
    / "morgan_features_84_263_UNIQUE"
)


# ============================================================
# INPUT FILES
# ============================================================

X_TRAIN_PATH = (
    FEATURE_DIR
    / "X_train.npz"
)

Y_TRAIN_PATH = (
    FEATURE_DIR
    / "Y_train.npy"
)


X_CAL_PATH = (
    FEATURE_DIR
    / "X_calibration.npz"
)

Y_CAL_PATH = (
    FEATURE_DIR
    / "Y_calibration.npy"
)


X_TEST_PATH = (
    FEATURE_DIR
    / "X_test.npz"
)

Y_TEST_PATH = (
    FEATURE_DIR
    / "Y_test.npy"
)


LABEL_MAP_PATH = (
    FEATURE_DIR
    / "label_mapping.csv"
)


# Stage 1.5 output uploaded directly in MyDrive
SELECTION_FILE = (
    MYDRIVE
    / "FINAL_label_parameter_selection.csv"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

FINAL_ROOT = (
    PROJECT_ROOT
    / "FINAL_UNIQUE263_HYBRID_LGBM"
)


OUT_DIR = (
    FINAL_ROOT
    / f"seed_{SEED}"
)


MODELS_DIR = (
    OUT_DIR
    / "models"
)


META_DIR = (
    OUT_DIR
    / "metadata"
)


MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


META_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5) BASIC INFO
# ============================================================

print("=" * 110)
print("STAGE 2 — FINAL HYBRID LIGHTGBM")
print("=" * 110)

print(
    "Seed:",
    SEED
)

print(
    "LightGBM:",
    lgb.__version__
)

print(
    "\nFeature directory:"
)

print(
    FEATURE_DIR
)

print(
    "\nSelection map:"
)

print(
    SELECTION_FILE
)

print(
    "\nOutput directory:"
)

print(
    OUT_DIR
)


# ============================================================
# 6) FILE CHECK
# ============================================================

paths = {

    "X_train":
        X_TRAIN_PATH,

    "Y_train":
        Y_TRAIN_PATH,

    "X_calibration":
        X_CAL_PATH,

    "Y_calibration":
        Y_CAL_PATH,

    "X_test":
        X_TEST_PATH,

    "Y_test":
        Y_TEST_PATH,

    "label_mapping":
        LABEL_MAP_PATH,

    "selection_map":
        SELECTION_FILE
}


print("\n" + "=" * 110)
print("FILE CHECK")
print("=" * 110)


missing = []


for name, path in paths.items():

    exists = path.exists()

    print(
        "" if exists else "",
        name,
        "->",
        path
    )

    if not exists:

        missing.append(
            name
        )


if missing:

    raise FileNotFoundError(
        f"\n Missing files: {missing}"
    )


print(
    "\n ALL REQUIRED FILES FOUND"
)


# ============================================================
# 7) LOAD FEATURES
# ============================================================

print("\n" + "=" * 110)
print("LOADING DATA")
print("=" * 110)


print(
    "\nLoading X_train..."
)

X_train = (
    sp.load_npz(
        X_TRAIN_PATH
    )
    .tocsr()
    .astype(
        np.float32
    )
)


print(
    "Loading X_calibration..."
)

X_cal = (
    sp.load_npz(
        X_CAL_PATH
    )
    .tocsr()
    .astype(
        np.float32
    )
)


print(
    "Loading X_test..."
)

X_test = (
    sp.load_npz(
        X_TEST_PATH
    )
    .tocsr()
    .astype(
        np.float32
    )
)


# ============================================================
# 8) LOAD TARGETS
# ============================================================

print(
    "Loading Y_train..."
)

Y_train = np.load(
    Y_TRAIN_PATH,
    mmap_mode="r"
)


print(
    "Loading Y_calibration..."
)

Y_cal = np.load(
    Y_CAL_PATH,
    mmap_mode="r"
)


print(
    "Loading Y_test..."
)

Y_test = np.load(
    Y_TEST_PATH,
    mmap_mode="r"
)


# ============================================================
# 9) LOAD LABEL MAPPING
# ============================================================

print(
    "Loading label mapping..."
)

label_map = (
    pd.read_csv(
        LABEL_MAP_PATH
    )
    .sort_values(
        "label_id"
    )
    .reset_index(
        drop=True
    )
)


LABELS = (
    label_map[
        "label"
    ]
    .astype(str)
    .tolist()
)


# ============================================================
# 10) LOAD STAGE 1.5 SELECTION MAP
# ============================================================

print(
    "Loading Stage 1.5 selection map..."
)

selection = (
    pd.read_csv(
        SELECTION_FILE
    )
    .sort_values(
        "label_id"
    )
    .reset_index(
        drop=True
    )
)


selection[
    "selected_parameter_set"
] = (
    selection[
        "selected_parameter_set"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ============================================================
# 11) DATA QA
# ============================================================

print("\n" + "=" * 110)
print("DATA QA")
print("=" * 110)


print(
    "X_train:",
    X_train.shape,
    X_train.dtype
)

print(
    "X_calibration:",
    X_cal.shape,
    X_cal.dtype
)

print(
    "X_test:",
    X_test.shape,
    X_test.dtype
)


print(
    "\nY_train:",
    Y_train.shape,
    Y_train.dtype
)

print(
    "Y_calibration:",
    Y_cal.shape,
    Y_cal.dtype
)

print(
    "Y_test:",
    Y_test.shape,
    Y_test.dtype
)


print(
    "\nLabels:",
    len(
        LABELS
    )
)

print(
    "Selection rows:",
    len(
        selection
    )
)


# ============================================================
# EXPECTED SHAPES
# ============================================================

assert X_train.shape == (
    134249,
    4096
), X_train.shape


assert X_cal.shape == (
    28768,
    4096
), X_cal.shape


assert X_test.shape == (
    28768,
    4096
), X_test.shape


assert Y_train.shape == (
    134249,
    263
), Y_train.shape


assert Y_cal.shape == (
    28768,
    263
), Y_cal.shape


assert Y_test.shape == (
    28768,
    263
), Y_test.shape


assert len(
    LABELS
) == 263


assert len(
    selection
) == 263


print(
    "\n All shapes are correct"
)


# ============================================================
# 12) VERIFY LABEL IDs
# ============================================================

expected_ids = np.arange(
    N_LABELS
)


actual_ids = (
    selection[
        "label_id"
    ]
    .astype(int)
    .values
)


assert np.array_equal(
    expected_ids,
    actual_ids
)


print(
    " Label IDs = 0..262"
)


# ============================================================
# 13) VERIFY LABEL TEXT ALIGNMENT
# ============================================================

if "label" in selection.columns:

    selection_labels = (
        selection[
            "label"
        ]
        .astype(str)
        .tolist()
    )


    mismatches = [

        i

        for i, (selection_label, mapping_label)

        in enumerate(
            zip(
                selection_labels,
                LABELS
            )
        )

        if selection_label
        !=
        mapping_label
    ]


    if mismatches:

        print(
            "\nFirst mismatches:"
        )

        for i in mismatches[:10]:

            print(
                i,
                selection_labels[i],
                "!=",
                LABELS[i]
            )


        raise RuntimeError(
            " Selection map labels do not match "
            "final 263-unique label mapping."
        )


print(
    " Label text alignment correct"
)


# ============================================================
# 14) VERIFY PARAMETER SELECTION
# ============================================================

allowed_sets = {
    "baseline",
    "tuned"
}


actual_sets = set(
    selection[
        "selected_parameter_set"
    ].unique()
)


assert actual_sets == allowed_sets


selection_counts = (
    selection[
        "selected_parameter_set"
    ]
    .value_counts()
    .to_dict()
)


print("\n" + "=" * 110)
print("STAGE 1.5 SELECTION")
print("=" * 110)


print(
    selection_counts
)


assert selection_counts.get(
    "baseline",
    0
) == 140


assert selection_counts.get(
    "tuned",
    0
) == 123


print(
    "\n Baseline labels = 140"
)

print(
    " Tuned labels    = 123"
)

print(
    " Stage 1.5 map correctly aligned"
)


# ============================================================
# 15) PARAMETER SETS
# ============================================================

BASELINE_PARAMS = {

    "n_estimators":
        500,

    "learning_rate":
        0.05,

    "num_leaves":
        63,

    "max_depth":
        -1,

    "min_child_samples":
        30,

    "subsample":
        0.85,

    "colsample_bytree":
        0.85,

    "reg_alpha":
        0.0,

    "reg_lambda":
        1.0
}


TUNED_PARAMS = {

    "n_estimators":
        900,

    "learning_rate":
        0.028630640852697415,

    "num_leaves":
        31,

    "max_depth":
        -1,

    "min_child_samples":
        100,

    "subsample":
        0.70,

    "colsample_bytree":
        0.70,

    "reg_alpha":
        0.0,

    "reg_lambda":
        0.6161362498757661
}


print("\n" + "=" * 110)
print("PARAMETER SETS")
print("=" * 110)

print(
    "\nBASELINE:"
)

print(
    BASELINE_PARAMS
)

print(
    "\nTUNED:"
)

print(
    TUNED_PARAMS
)


# ============================================================
# 16) SAVE EXACT CONFIG USED
# ============================================================

selection.to_csv(
    OUT_DIR
    / "selection_map_used.csv",
    index=False
)


training_config = {

    "seed":
        SEED,

    "lightgbm_version":
        lgb.__version__,

    "n_labels":
        N_LABELS,

    "feature_size":
        4096,

    "early_stopping_rounds":
        EARLY_STOPPING_ROUNDS,

    "baseline_label_count":
        140,

    "tuned_label_count":
        123,

    "baseline_parameters":
        BASELINE_PARAMS,

    "tuned_parameters":
        TUNED_PARAMS,

    "feature_directory":
        str(
            FEATURE_DIR
        ),

    "selection_file":
        str(
            SELECTION_FILE
        ),

    "test_used_for_parameter_selection":
        False,

    "test_metrics_computed_during_training":
        False
}


with open(
    OUT_DIR
    / "training_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        training_config,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 17) CREATE PROBABILITY ARRAYS
# ============================================================

cal_probabilities = np.zeros(
    (
        X_cal.shape[0],
        N_LABELS
    ),
    dtype=np.float32
)


test_probabilities = np.zeros(
    (
        X_test.shape[0],
        N_LABELS
    ),
    dtype=np.float32
)


# ============================================================
# 18) TRAIN / RESUME
# ============================================================

print("\n" + "=" * 110)
print("STARTING SEED 44 TRAINING")
print("=" * 110)


training_start = time.time()


trained_now = 0

loaded_existing = 0


for label_id in range(
    N_LABELS
):


    # ========================================================
    # LABEL INFO
    # ========================================================

    label_name = (
        LABELS[
            label_id
        ]
    )


    selected_set = (
        selection.loc[
            label_id,
            "selected_parameter_set"
        ]
    )


    if selected_set == "baseline":

        chosen_params = dict(
            BASELINE_PARAMS
        )


    elif selected_set == "tuned":

        chosen_params = dict(
            TUNED_PARAMS
        )


    else:

        raise ValueError(
            f"Invalid parameter set "
            f"for label {label_id}: "
            f"{selected_set}"
        )


    # ========================================================
    # OUTPUT FILES
    # ========================================================

    model_path = (
        MODELS_DIR
        / f"label_{label_id:03d}.txt"
    )


    meta_path = (
        META_DIR
        / f"label_{label_id:03d}.json"
    )


    print("\n" + "=" * 100)

    print(
        f"{label_id + 1:03d}/263"
        f" | ID={label_id:03d}"
        f" | {label_name}"
    )

    print(
        "Parameter set:",
        selected_set.upper()
    )


    # ========================================================
    # RESUME EXISTING COMPLETE MODEL
    # ========================================================

    if (
        model_path.exists()
        and
        meta_path.exists()
    ):

        print(
            " Existing model + metadata found"
        )

        print(
            "Loading instead of retraining..."
        )


        with open(
            meta_path,
            "r",
            encoding="utf-8"
        ) as f:

            meta = json.load(
                f
            )


        if int(
            meta["seed"]
        ) != SEED:

            raise RuntimeError(
                f"Seed mismatch for label "
                f"{label_id}"
            )


        if (
            meta[
                "selected_parameter_set"
            ]
            !=
            selected_set
        ):

            raise RuntimeError(
                f"Parameter-set mismatch "
                f"for label {label_id}"
            )


        booster = lgb.Booster(
            model_file=str(
                model_path
            )
        )


        best_iteration = int(
            meta[
                "best_iteration"
            ]
        )


        # Calibration probabilities
        cal_probabilities[
            :,
            label_id
        ] = booster.predict(

            X_cal,

            num_iteration=
                best_iteration

        ).astype(
            np.float32
        )


        # Test probabilities
        test_probabilities[
            :,
            label_id
        ] = booster.predict(

            X_test,

            num_iteration=
                best_iteration

        ).astype(
            np.float32
        )


        loaded_existing += 1


        print(
            f" Loaded"
            f" | best_iter={best_iteration}"
        )


        del booster

        gc.collect()

        continue


    # ========================================================
    # TARGETS
    # ========================================================

    y_train = np.asarray(
        Y_train[
            :,
            label_id
        ],
        dtype=np.int32
    )


    y_cal = np.asarray(
        Y_cal[
            :,
            label_id
        ],
        dtype=np.int32
    )


    train_pos = int(
        y_train.sum()
    )


    train_neg = int(
        len(
            y_train
        )
        -
        train_pos
    )


    cal_pos = int(
        y_cal.sum()
    )


    if train_pos == 0:

        raise RuntimeError(
            f"Label {label_id} "
            "has zero Train positives."
        )


    if cal_pos == 0:

        raise RuntimeError(
            f"Label {label_id} "
            "has zero Calibration positives."
        )


    # ========================================================
    # SCALE POS WEIGHT
    # ========================================================

    scale_pos_weight = float(

        np.sqrt(

            (
                train_neg
                +
                1
            )

            /

            (
                train_pos
                +
                1
            )

        )

    )


    print(
        "Train positives:",
        train_pos
    )

    print(
        "Calibration positives:",
        cal_pos
    )

    print(
        "scale_pos_weight:",
        round(
            scale_pos_weight,
            6
        )
    )


    # ========================================================
    # FINAL MODEL PARAMETERS
    # ========================================================

    model_params = dict(
        chosen_params
    )


    model_params.update({

        "objective":
            "binary",

        "verbosity":
            -1,

        "n_jobs":
            N_JOBS,

        "random_state":
            SEED
            +
            int(
                label_id
            ),

        "scale_pos_weight":
            scale_pos_weight,

        "subsample_freq":
            1
    })


    model = lgb.LGBMClassifier(
        **model_params
    )


    label_start = time.time()


    # ========================================================
    # TRAIN
    # ========================================================

    model.fit(

        X_train,

        y_train,

        eval_set=[
            (
                X_cal,
                y_cal
            )
        ],

        eval_metric=
            "binary_logloss",

        callbacks=[

            lgb.early_stopping(

                stopping_rounds=
                    EARLY_STOPPING_ROUNDS,

                verbose=False

            )

        ]
    )


    elapsed = (
        time.time()
        -
        label_start
    )


    # ========================================================
    # BEST ITERATION
    # ========================================================

    if model.best_iteration_:

        best_iteration = int(
            model.best_iteration_
        )

    else:

        best_iteration = int(
            chosen_params[
                "n_estimators"
            ]
        )


    # ========================================================
    # SAVE MODEL
    # ========================================================

    model.booster_.save_model(

        str(
            model_path
        ),

        num_iteration=
            best_iteration
    )


    # ========================================================
    # CALIBRATION PROBABILITIES
    # ========================================================

    cal_probabilities[
        :,
        label_id
    ] = (

        model.predict_proba(

            X_cal,

            num_iteration=
                best_iteration

        )[:, 1]

        .astype(
            np.float32
        )

    )


    # ========================================================
    # TEST PROBABILITIES
    #
    # SAVE ONLY.
    # NO TEST METRICS.
    # ========================================================

    test_probabilities[
        :,
        label_id
    ] = (

        model.predict_proba(

            X_test,

            num_iteration=
                best_iteration

        )[:, 1]

        .astype(
            np.float32
        )

    )


    # ========================================================
    # METADATA
    # ========================================================

    metadata = {

        "label_id":
            int(
                label_id
            ),

        "label":
            label_name,

        "seed":
            SEED,

        "selected_parameter_set":
            selected_set,

        "selected_parameters":
            chosen_params,

        "train_support":
            train_pos,

        "calibration_support":
            cal_pos,

        "scale_pos_weight":
            scale_pos_weight,

        "best_iteration":
            best_iteration,

        "training_seconds":
            float(
                elapsed
            ),

        "test_metrics_used":
            False
    }


    with open(
        meta_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False
        )


    trained_now += 1


    print(
        " Finished"
    )

    print(
        "best_iteration:",
        best_iteration
    )

    print(
        "seconds:",
        round(
            elapsed,
            1
        )
    )


    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del model
    del y_train
    del y_cal

    gc.collect()


    # ========================================================
    # PARTIAL BACKUP EVERY 10 LABELS
    # ========================================================

    if (
        (
            label_id
            +
            1
        )
        %
        10
        ==
        0
    ):

        np.save(
            OUT_DIR
            / "calibration_probabilities_PARTIAL.npy",

            cal_probabilities
        )


        np.save(
            OUT_DIR
            / "test_probabilities_PARTIAL.npy",

            test_probabilities
        )


        print(
            " Partial probability arrays saved"
        )


# ============================================================
# 19) SAVE FINAL PROBABILITIES
# ============================================================

CAL_PROB_PATH = (
    OUT_DIR
    / "calibration_probabilities.npy"
)


TEST_PROB_PATH = (
    OUT_DIR
    / "test_probabilities.npy"
)


np.save(
    CAL_PROB_PATH,
    cal_probabilities
)


np.save(
    TEST_PROB_PATH,
    test_probabilities
)


# ============================================================
# 20) FINAL MODEL FILE QA
# ============================================================

model_files = sorted(
    MODELS_DIR.glob(
        "label_*.txt"
    )
)


meta_files = sorted(
    META_DIR.glob(
        "label_*.json"
    )
)


print("\n" + "=" * 110)
print("FINAL QA")
print("=" * 110)


print(
    "Models:",
    len(
        model_files
    ),
    "/263"
)


print(
    "Metadata:",
    len(
        meta_files
    ),
    "/263"
)


print(
    "Calibration probabilities:",
    cal_probabilities.shape
)


print(
    "Test probabilities:",
    test_probabilities.shape
)


assert len(
    model_files
) == 263


assert len(
    meta_files
) == 263


assert cal_probabilities.shape == (
    28768,
    263
)


assert test_probabilities.shape == (
    28768,
    263
)


assert np.isfinite(
    cal_probabilities
).all()


assert np.isfinite(
    test_probabilities
).all()


assert (
    cal_probabilities.min()
    >=
    0
)


assert (
    cal_probabilities.max()
    <=
    1
)


assert (
    test_probabilities.min()
    >=
    0
)


assert (
    test_probabilities.max()
    <=
    1
)


print(
    "\n Probability arrays passed QA"
)


# ============================================================
# 21) TRAINING COMPLETE MARKER
# ============================================================

total_seconds = (
    time.time()
    -
    training_start
)


completion = {

    "seed":
        SEED,

    "status":
        "complete",

    "n_labels":
        263,

    "baseline_labels":
        140,

    "tuned_labels":
        123,

    "models":
        len(
            model_files
        ),

    "metadata":
        len(
            meta_files
        ),

    "trained_now":
        trained_now,

    "loaded_existing":
        loaded_existing,

    "calibration_probability_shape":
        list(
            cal_probabilities.shape
        ),

    "test_probability_shape":
        list(
            test_probabilities.shape
        ),

    "total_seconds_this_run":
        float(
            total_seconds
        ),

    "test_metrics_computed":
        False
}


with open(
    OUT_DIR
    / "training_complete.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        completion,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 22) REMOVE PARTIAL ARRAYS AFTER SUCCESS
# ============================================================

partial_files = [

    OUT_DIR
    / "calibration_probabilities_PARTIAL.npy",

    OUT_DIR
    / "test_probabilities_PARTIAL.npy"
]


for path in partial_files:

    if path.exists():

        path.unlink()


# ============================================================
# 23) FINAL OUTPUT
# ============================================================

print("\n" + "=" * 110)
print("SEED 44 COMPLETE")
print("=" * 110)


print(
    "Trained this run:",
    trained_now
)


print(
    "Loaded existing:",
    loaded_existing
)


print(
    "Models:",
    len(
        model_files
    ),
    "/263"
)


print(
    "\nOutput folder:"
)

print(
    OUT_DIR
)


print(
    "\nCalibration probabilities:"
)

print(
    CAL_PROB_PATH
)


print(
    "\nTest probabilities:"
)

print(
    TEST_PROB_PATH
)


print(
    "\nCompletion file:"
)

print(
    OUT_DIR
    / "training_complete.json"
)


print(
    "\n SEED 44 STAGE 2 FINISHED SUCCESSFULLY"
)


print(
    " Test probabilities were generated, "
    "but NO test metrics were calculated."
)